In [129]:
%pip install  openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [130]:
import pandas as pd
import openpyxl
from scipy.stats import chi2_contingency


In [131]:
from statsmodels.stats.outliers_influence import variance_inflation_factor



In [132]:
df1=pd.read_excel(r"C:\Users\Lenovo\Downloads\credit_card\case_study1.xlsx")
df2=pd.read_excel(r"C:\Users\Lenovo\Downloads\credit_card\case_study2.xlsx")


In [133]:
df1.columns

Index(['PROSPECTID', 'Total_TL', 'Tot_Closed_TL', 'Tot_Active_TL',
       'Total_TL_opened_L6M', 'Tot_TL_closed_L6M', 'pct_tl_open_L6M',
       'pct_tl_closed_L6M', 'pct_active_tl', 'pct_closed_tl',
       'Total_TL_opened_L12M', 'Tot_TL_closed_L12M', 'pct_tl_open_L12M',
       'pct_tl_closed_L12M', 'Tot_Missed_Pmnt', 'Auto_TL', 'CC_TL',
       'Consumer_TL', 'Gold_TL', 'Home_TL', 'PL_TL', 'Secured_TL',
       'Unsecured_TL', 'Other_TL', 'Age_Oldest_TL', 'Age_Newest_TL'],
      dtype='object')

In [134]:
df1=df1[df1['Age_Oldest_TL']!=-99999]

In [135]:
df1.shape,df2.shape

((51296, 26), (51336, 62))

In [ ]:
def clean_mostly_null_cols(df):
    a=[]
    for i in df.columns:
        if df2[df2[i]==-99999].shape[0]>10000:
            a.append(i)
    return df.drop(a,axis=1)

In [ ]:
def clean_mostly_null_rows(df):
    for i in df.columns:
        df=df[df['i']!=-99999]
    return df

    

In [136]:
a=[]
for i in df2.columns:
    if df2[df2[i]==-99999].shape[0]>10000:
        a.append(i)
    

In [137]:
df2=df2.drop(a,axis=1)

In [138]:
df2.shape

(51336, 54)

In [139]:
for i in df2.columns:
    df2=df2[df2[i]!=-99999]

In [140]:
for i in df1.columns.tolist():
    if i in df2.columns.tolist():
        print(i)

PROSPECTID


In [141]:
df=df1.merge(df2,on='PROSPECTID',how='inner')

In [ ]:
def chi_suqure_test_removal_cols(df,target):
    ##only valid when your .. target is object or cat columns
    object_arr=[]
    cols_to_remove=[]
    df1=df.drop(target,axis=1)
    for i in df1.columns:
        if df1[i].dtype=='object':
                object_arr.append(i)
    for i in object_arr:
         chi2,pval,_,_=chi2_contingency(pd.crosstab(df[i],df[target]))
         if pval>0.05:
              cols_to_remove.append(i)
    return df.drop(cols_to_remove,axis=1),object_arr
         
        



In [142]:
##chi sqaure test in order to see is catgeory column in fetaure reated with target column or not
##find the catgeory columns
object_arr=[]
for i in df.columns:
    if df[i].dtype=='object':
        object_arr.append(i)


In [143]:
object_arr

['MARITALSTATUS',
 'EDUCATION',
 'GENDER',
 'last_prod_enq2',
 'first_prod_enq2',
 'Approved_Flag']

In [144]:
##Applying.. chi_square_test
for i in ['MARITALSTATUS',
 'EDUCATION',
 'GENDER',
 'last_prod_enq2',
 'first_prod_enq2']:
    chi2,pval,_,_=chi2_contingency(pd.crosstab(df[i],df['Approved_Flag']))
    print(i,'-----',pval)

MARITALSTATUS ----- 3.578180861038862e-233
EDUCATION ----- 2.6942265249737532e-30
GENDER ----- 1.907936100186563e-05
last_prod_enq2 ----- 0.0
first_prod_enq2 ----- 7.84997610555419e-287


In [145]:
##now we have to .. check for vif .. since we are using logisitc regression
numeric_cols=[]
for i in df.columns:
    if df[i].dtype!='object' and i not in ['Approved_Flag','PROSPECTID']:
        numeric_cols.append(i)

In [ ]:
def column_filter_based_on_vif_and_annova(df,target):
    numeric_cols=[]
    for i in df.columns:
        if df[i].dtype!='object' and i not in ['Approved_Flag','PROSPECTID']:
            numeric_cols.append(i)
    vif_data=df[numeric_cols]
    total_columns=vif_data.shape[1]
    columns_to_kept=[]
    columns_index=0
    for i in range(0,total_columns):
        vif_value=variance_inflation_factor(vif_data,columns_index)
        print(vif_value,'___',columns_index)
        if vif_value<=6:
            columns_to_kept.append(numeric_cols[i])
            columns_index=columns_index+1
        else:
            vif_data=vif_data.drop([numeric_cols[i]],axis=1)
    annova_result={}
    columns_to_kept_in_numerical=[]
    for i in columns_to_kept:
        groups=[df[df['Approved_Flag']==cat][i].dropna() for cat in df['Approved_Flag'].unique()]
        f_stat,p_value=f_oneway(*groups)
        annova_result[i]=p_value
    anova_df = pd.DataFrame(list(annova_result.items()), columns=['feature', 'p_value'])
    anova_df = anova_df.sort_values('p_value')
    return anova_df[anova_df['p_value']<=0.05]['feature'].unique().tolist()

In [146]:
numeric_cols

['Total_TL',
 'Tot_Closed_TL',
 'Tot_Active_TL',
 'Total_TL_opened_L6M',
 'Tot_TL_closed_L6M',
 'pct_tl_open_L6M',
 'pct_tl_closed_L6M',
 'pct_active_tl',
 'pct_closed_tl',
 'Total_TL_opened_L12M',
 'Tot_TL_closed_L12M',
 'pct_tl_open_L12M',
 'pct_tl_closed_L12M',
 'Tot_Missed_Pmnt',
 'Auto_TL',
 'CC_TL',
 'Consumer_TL',
 'Gold_TL',
 'Home_TL',
 'PL_TL',
 'Secured_TL',
 'Unsecured_TL',
 'Other_TL',
 'Age_Oldest_TL',
 'Age_Newest_TL',
 'time_since_recent_payment',
 'num_times_delinquent',
 'max_recent_level_of_deliq',
 'num_deliq_6mts',
 'num_deliq_12mts',
 'num_deliq_6_12mts',
 'num_times_30p_dpd',
 'num_times_60p_dpd',
 'num_std',
 'num_std_6mts',
 'num_std_12mts',
 'num_sub',
 'num_sub_6mts',
 'num_sub_12mts',
 'num_dbt',
 'num_dbt_6mts',
 'num_dbt_12mts',
 'num_lss',
 'num_lss_6mts',
 'num_lss_12mts',
 'recent_level_of_deliq',
 'tot_enq',
 'CC_enq',
 'CC_enq_L6m',
 'CC_enq_L12m',
 'PL_enq',
 'PL_enq_L6m',
 'PL_enq_L12m',
 'time_since_recent_enq',
 'enq_L12m',
 'enq_L6m',
 'enq_L3m',

In [147]:
vif_data=df[numeric_cols]
total_columns=vif_data.shape[1]
columns_to_kept=[]
columns_index=0
for i in range(0,total_columns):
    vif_value=variance_inflation_factor(vif_data,columns_index)
    print(vif_value,'___',columns_index)
    if vif_value<=6:
        columns_to_kept.append(numeric_cols[i])
        columns_index=columns_index+1
    else:
        vif_data=vif_data.drop([numeric_cols[i]],axis=1)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22796\3493903427.py:6: UserWarning: The design matrix is poorly conditioned (condition number=2.05e+16). VIF calculations may be numerically unstable.
  vif_value=variance_inflation_factor(vif_data,columns_index)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


83976641.25592071 ___ 0


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22796\3493903427.py:6: UserWarning: The design matrix is poorly conditioned (condition number=5.80e+15). VIF calculations may be numerically unstable.
  vif_value=variance_inflation_factor(vif_data,columns_index)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


2302679.1319180964 ___ 0


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22796\3493903427.py:6: UserWarning: The design matrix is poorly conditioned (condition number=7.36e+15). VIF calculations may be numerically unstable.
  vif_value=variance_inflation_factor(vif_data,columns_index)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


...

2.7209525456967274 ___ 15


C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


6.909892271167477 ___ 16


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22796\3493903427.py:6: UserWarning: The design matrix is poorly conditioned (condition number=3.95e+15). VIF calculations may be numerically unstable.
  vif_value=variance_inflation_factor(vif_data,columns_index)
C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


4.921457561232886 ___ 16
1000799917193443.5 ___ 17
6.64060783017617 ___ 17
1.3159154707874672 ___ 17
7.625456477063645 ___ 18
1.5813808513856031 ___ 18
6.247287793745863 ___ 19
13.461183034059781 ___ 19
2.1153094307816485 ___ 19
1.499535519387001 ___ 20
2.1707974597343638 ___ 21
2.6194659619474043 ___ 22
2.2938993517650528 ___ 23
7.358623292167585 ___ 24
2.159662759661745 ___ 24
2.866271116689629 ___ 25
6.4569247347387195 ___ 26
2.847110953867579 ___ 26
4.498908692151748 ___ 27
9.682605341254286 ___ 28
5.967208313487437 ___ 28
8.495827332703996 ___ 29
5.15208436161069 ___ 29
7.280751669212345 ___ 30
11.481328638663035 ___ 30
2.967507642896107 ___ 30
1.595005990346598 ___ 31
12.530922827011443 ___ 32
7.836355832427285 ___ 32
3.066874602451977 ___ 32
1.4041913778107653 ___ 33
1.0484933703163484 ___ 34
1.2675679133867643 ___ 35
3.1270620692109894 ___ 36
4.340308450459927 ___ 37
1.000802207102392 ___ 38
2.8456129815466973 ___ 39
2.2794120029683653 ___ 40
15.827331223869102 ___ 41
16.387029

In [148]:
columns_to_kept

['pct_tl_open_L6M',
 'pct_tl_closed_L6M',
 'Total_TL_opened_L12M',
 'Tot_TL_closed_L12M',
 'pct_tl_open_L12M',
 'pct_tl_closed_L12M',
 'Tot_Missed_Pmnt',
 'CC_TL',
 'Home_TL',
 'PL_TL',
 'Secured_TL',
 'Unsecured_TL',
 'Other_TL',
 'Age_Oldest_TL',
 'Age_Newest_TL',
 'time_since_recent_payment',
 'max_recent_level_of_deliq',
 'num_deliq_6_12mts',
 'num_times_60p_dpd',
 'num_std_12mts',
 'num_sub',
 'num_sub_6mts',
 'num_sub_12mts',
 'num_dbt',
 'num_dbt_12mts',
 'num_lss',
 'num_lss_12mts',
 'recent_level_of_deliq',
 'CC_enq',
 'CC_enq_L12m',
 'PL_enq_L12m',
 'time_since_recent_enq',
 'enq_L3m',
 'AGE',
 'NETMONTHLYINCOME',
 'Time_With_Curr_Empr',
 'pct_of_active_TLs_ever',
 'pct_opened_TLs_L6m_of_L12m',
 'pct_currentBal_all_TL',
 'CC_Flag',
 'PL_Flag',
 'pct_PL_enq_L6m_of_ever',
 'pct_CC_enq_L6m_of_ever',
 'HL_Flag',
 'GL_Flag',
 'Credit_Score']

In [149]:
##for the columns which kept check Annova for them
from scipy.stats import  f_oneway
annova_result={}
columns_to_kept_in_numerical=[]
for i in columns_to_kept:
    groups=[df[df['Approved_Flag']==cat][i].dropna() for cat in df['Approved_Flag'].unique()]
    f_stat,p_value=f_oneway(*groups)
    annova_result[i]=p_value
            

In [150]:
anova_df = pd.DataFrame(list(annova_result.items()), columns=['feature', 'p_value'])
anova_df = anova_df.sort_values('p_value')
print(anova_df)

                       feature        p_value
0              pct_tl_open_L6M   0.000000e+00
4             pct_tl_open_L12M   0.000000e+00
10                  Secured_TL   0.000000e+00
8                      Home_TL   0.000000e+00
12                    Other_TL   0.000000e+00
13               Age_Oldest_TL   0.000000e+00
29                 CC_enq_L12m   0.000000e+00
19               num_std_12mts   0.000000e+00
30                 PL_enq_L12m   0.000000e+00
31       time_since_recent_enq   0.000000e+00
45                Credit_Score   0.000000e+00
44                     GL_Flag   0.000000e+00
41      pct_PL_enq_L6m_of_ever   0.000000e+00
42      pct_CC_enq_L6m_of_ever   0.000000e+00
33                         AGE   0.000000e+00
32                     enq_L3m   0.000000e+00
35         Time_With_Curr_Empr  6.743540e-271
36      pct_of_active_TLs_ever  1.501231e-270
43                     HL_Flag  1.132780e-237
9                        PL_TL  7.991892e-235
6              Tot_Missed_Pmnt  9.

In [151]:
columns_to_kept_in_numerical.append(anova_df[anova_df['p_value']<=0.05]['feature'].unique())

In [152]:
columns_to_kept_in_numerical

[array(['pct_tl_open_L6M', 'pct_tl_open_L12M', 'Secured_TL', 'Home_TL',
        'Other_TL', 'Age_Oldest_TL', 'CC_enq_L12m', 'num_std_12mts',
        'PL_enq_L12m', 'time_since_recent_enq', 'Credit_Score', 'GL_Flag',
        'pct_PL_enq_L6m_of_ever', 'pct_CC_enq_L6m_of_ever', 'AGE',
        'enq_L3m', 'Time_With_Curr_Empr', 'pct_of_active_TLs_ever',
        'HL_Flag', 'PL_TL', 'Tot_Missed_Pmnt', 'CC_enq',
        'pct_opened_TLs_L6m_of_L12m', 'PL_Flag', 'Total_TL_opened_L12M',
        'Unsecured_TL', 'Tot_TL_closed_L12M', 'recent_level_of_deliq',
        'Age_Newest_TL', 'max_recent_level_of_deliq',
        'time_since_recent_payment', 'num_deliq_6_12mts',
        'num_times_60p_dpd', 'CC_TL', 'pct_tl_closed_L6M', 'CC_Flag',
        'pct_tl_closed_L12M', 'NETMONTHLYINCOME', 'num_sub', 'num_dbt',
        'num_sub_12mts', 'num_dbt_12mts', 'num_lss', 'num_sub_6mts'],
       dtype=object)]

In [153]:
feature=anova_df[anova_df['p_value']<=0.05]['feature'].unique().tolist()+['MARITALSTATUS',
 'EDUCATION',
 'GENDER',
 'last_prod_enq2',
 'first_prod_enq2']

In [ ]:
feature=column_filter_based_on_vif_and_annova(df,'Approved_Flag')
feature=feature+

In [154]:
# in main_file.ipynb, wherever you finalize your feature list before df_encoded:
feature = [f for f in feature if f != 'Credit_Score']


In [155]:
df=df[feature+['Approved_Flag']]

In [156]:
for i in ['MARITALSTATUS',
 'EDUCATION',
 'GENDER',
 'last_prod_enq2',
 'first_prod_enq2']:
    print(df[i].nunique(),i)

2 MARITALSTATUS
7 EDUCATION
2 GENDER
6 last_prod_enq2
6 first_prod_enq2


In [157]:
#excpet edduavtion we are going to create dummies for everyone
#for education
df['EDUCATION'].unique()

array(['12TH', 'GRADUATE', 'SSC', 'POST-GRADUATE', 'UNDER GRADUATE',
       'OTHERS', 'PROFESSIONAL'], dtype=object)

In [158]:
education_level_map = {
    'SSC': 1,
    '12TH': 2,
    'UNDER GRADUATE': 3,
    'GRADUATE': 3,
    'PROFESSIONAL': 3,
    'POST-GRADUATE': 4,
    'OTHERS': 1
}

df['EDUCATION'] = df['EDUCATION'].map(education_level_map)


In [159]:
df['EDUCATION']=df['EDUCATION'].astype('int64')

In [160]:
df_encoded=pd.get_dummies(df,columns=['MARITALSTATUS',
 'EDUCATION',
 'GENDER',
 'last_prod_enq2',
 'first_prod_enq2'])

In [161]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 64 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   pct_tl_open_L6M               42064 non-null  float64
 1   pct_tl_open_L12M              42064 non-null  float64
 2   Secured_TL                    42064 non-null  int64  
 3   Home_TL                       42064 non-null  int64  
 4   Other_TL                      42064 non-null  int64  
 5   Age_Oldest_TL                 42064 non-null  int64  
 6   CC_enq_L12m                   42064 non-null  int64  
 7   num_std_12mts                 42064 non-null  int64  
 8   PL_enq_L12m                   42064 non-null  int64  
 9   time_since_recent_enq         42064 non-null  int64  
 10  GL_Flag                       42064 non-null  int64  
 11  pct_PL_enq_L6m_of_ever        42064 non-null  float64
 12  pct_CC_enq_L6m_of_ever        42064 non-null  float64
 13  A

In [162]:
k=df_encoded.describe()

In [163]:
k

,pct_tl_open_L6M,pct_tl_open_L12M,Secured_TL,Home_TL,Other_TL,Age_Oldest_TL,CC_enq_L12m,num_std_12mts,PL_enq_L12m,time_since_recent_enq,...,pct_tl_closed_L6M,CC_Flag,pct_tl_closed_L12M,NETMONTHLYINCOME,num_sub,num_dbt,num_sub_12mts,num_dbt_12mts,num_lss,num_sub_6mts
count,42064.000000,42064.000000,42064.000000,42064.000000,42064.000000,42064.000000,42064.000000,42064.000000,42064.000000,42064.000000,...,42064.000000,42064.000000,42064.000000,4.206400e+04,42064.000000,42064.00000,42064.000000,42064.000000,42064.000000,42064.000000
mean,0.179032,0.401271,2.921334,0.076241,1.116489,46.498074,0.268924,3.279978,0.779194,264.854507,...,0.097783,0.102962,0.160365,2.692990e+04,0.063831,0.02451,0.009224,0.004279,0.016713,0.002211
std,0.278043,0.381266,6.379764,0.358582,2.486801,42.109230,1.019459,7.566312,1.802092,466.585002,...,0.210957,0.303913,0.258831,2.084300e+04,0.799989,0.62189,0.220786,0.184461,0.573762,0.081704
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,0.000000,0.000000,0.000000,9.000000,...,0.000000,0.000000,0.000000,1.800000e+04,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.333000,1.000000,0.000000,0.000000,34.000000,0.000000,0.000000,0.000000,79.000000,...,0.000000,0.000000,0.000000,2.400000e+04,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
75%,0.333000,0.714000,3.000000,0.000000,1.000000,65.000000,0.000000,3.000000,1.000000,302.000000,...,0.100000,0.000000,0.250000,3.100000e+04,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,235.000000,10.000000,80.000000,385.000000,24.000000,122.000000,44.000000,4768.000000,...,1.000000,1.000000,1.000000,2.500000e+06,41.000000,35.00000,12.000000,12.000000,72.000000,5.000000


In [164]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [165]:
rf=RandomForestClassifier(n_estimators=200,random_state=42)

In [166]:
y=df_encoded['Approved_Flag']
X=df_encoded.drop(['Approved_Flag'],axis=1)

In [167]:
from sklearn.preprocessing import LabelEncoder
label_encoder=LabelEncoder()
y_encoded=label_encoder.fit_transform(y)

In [168]:
X_train,X_test,y_train,y_test=train_test_split(X,y_encoded,test_size=0.2,random_state=42)

In [169]:
rf.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [170]:
y_pred=rf.predict(X_test)

In [171]:
from sklearn.metrics import accuracy_score,classification_report,precision_recall_fscore_support

In [172]:
print(accuracy_score(y_test,y_pred))

0.7625104005705455


In [175]:
precision,recall,f1_score,_=precision_recall_fscore_support(y_test,y_pred)
for i,v in enumerate(['p1','p2','p3','p4']):
    print(f"class{v}")
    print(f"precision{precision[i]}")
    print(f"recall{recall[i]}")
    print(f"f1 score{f1_score[i]}")

classp1
precision0.8552147239263803
recall0.6873767258382643
f1 score0.762165117550574
classp2
precision0.7878939469734867
recall0.9365708622398414
f1 score0.8558232204310813
classp3
precision0.4342560553633218
recall0.18943396226415093
f1 score0.26379400945874937
classp4
precision0.7253176930596286
recall0.7210884353741497
f1 score0.723196881091618
